# Datamine TDOUT Process Example

This notebook demonstrates how to configure and run the **`tdout`** process wrapper in `dmstudio`.

## Process Description

## TDOUT

Processes **TDOUT** and [TDIN](<tdin.md>) provide the interface between your application and the standalone Whittle THREE-D program for pit Optimization.

Process **TDOUT** takes a Datamine model and creates two ascii output files, which can be read by THREE-D. When THREE-D is run an output results file is created. Process [TDIN](<tdin.md>) reads this results file, and the associated parameter file, and creates a Datamine model:

![](../Images/tdout1.gif)

In ther words, Studio generates external economic and parameter files, and reads external result and parameter files.

The THREE-D economic file has a record per block with the following information:

* Block index

* Block value

* Ore-body identification

The block dimensions are defined by the Datamine model cell structure. If the model has subcells the economic value for the block is found by accumulating all the subcell values. Where the accumulated subcell volume is less than the cell volume, the remainder will be estimated with the default parameter VDEFAULT.

The ore-body identification is defined by a numeric field. If a subcell model is supplied, the dominant field value is output. The information is used when printing plans and sections of results files, or to display the position of the orebody for checking purposes. Ore-body identification numbers can be in the range 0-61 to be printed with values 0-9,A-Z,a-z.

If a subcell model is supplied, it must be in sorted (IJK) order.

### Input Files:

* **in** (Block Model):
  Input model file.
  Required=Yes

### Output Files:

### Fields:

* **value** (Numeric : IN):
  Economic value field.
  Default=Undefined
  Required=Yes

* **zone** (Any : IN):
  Ore-body identification field.
  Default=Undefined
  Required=No

### Parameters:

* **vdefault**:
  Default block value per unit volume (0.0).
  Range=Undefined
  Values=Undefined
  Default=0.0
  Required=No

* **format**:
  Output format for economic file (0). 0 - fixed format 1 \- comma separated
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('tdout')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute tdout
print("Running tdout...")
dm_cmd.tdout(
    in_i='t_assays',  # required
    value_f='optional',  # required
    # zone_f='optional',  # optional
    # vdefault_p=0.0,  # optional
    # format_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("tdout execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_tdout_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")